# Exercises

In this section we have two exercises:
1. Implement the polynomial kernel.
2. Implement the multiclass C-SVM.

## Polynomial kernel

You need to extend the ``build_kernel`` function and implement the polynomial kernel if the ``kernel_type`` is set to 'poly'. The equation that needs to be implemented:
\begin{equation}
K=(X^{T}*Y)^{d}.
\end{equation}

In [2]:
def build_kernel(data_set, kernel_type='linear'):
    kernel = np.dot(data_set, data_set.T)
    if kernel_type == 'rbf':
        sigma = 1.0
        objects_count = len(data_set)
        b = np.ones((len(data_set), 1))
        kernel -= 0.5 * (np.dot((np.diag(kernel)*np.ones((1, objects_count))).T, b.T)
                         + np.dot(b, (np.diag(kernel) * np.ones((1, objects_count))).T.T))
        kernel = np.exp(kernel / (2. * sigma ** 2))
    elif kernel_type == 'poly':
        d = 2
        kernel = kernel ** d
    return kernel

## Implement a multiclass C-SVM

Use the classification method that we used in notebook 7.3 and IRIS dataset to build a multiclass C-SVM classifier. Most implementation is about a function that will return the proper data set that need to be used for the prediction. You need to implement:
- ``choose_set_for_label``
- ``get_labels_count``

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_iris
import cvxopt
import numpy as np

def choose_set_for_label(data_set, label):
    X = data_set[:, :-1]
    y = data_set[:, -1]
    
    train_data_set, test_data_set, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    train_labels = np.where(y_train == label, 1, -1).astype(float).reshape(-1, 1)
    test_labels = np.where(y_test == label, 1, -1).astype(float).reshape(-1, 1)
    return train_data_set, test_data_set, train_labels, test_labels

In [4]:
def get_labels_count(data_set):
    return len(np.unique(data_set[:, -1]))

In [ ]:
# Use the code that we have implemented earlier:

In [8]:
def train(train_data_set, train_labels, kernel_type='linear', C=10, threshold=1e-5):
    kernel = build_kernel(train_data_set, kernel_type=kernel_type)

    P = train_labels * train_labels.transpose() * kernel
    q = -np.ones((objects_count, 1))
    G = np.concatenate((np.eye(objects_count), -np.eye(objects_count)))
    h = np.concatenate((C * np.ones((objects_count, 1)), np.zeros((objects_count, 1))))

    A = train_labels.reshape(1, objects_count)
    A = A.astype(float)
    b = 0.0

    sol = cvxopt.solvers.qp(cvxopt.matrix(P), cvxopt.matrix(q), cvxopt.matrix(G), cvxopt.matrix(h), cvxopt.matrix(A), cvxopt.matrix(b))

    lambdas = np.array(sol['x'])

    support_vectors_id = np.where(lambdas > threshold)[0]
    vector_number = len(support_vectors_id)
    support_vectors = train_data_set[support_vectors_id, :]

    lambdas = lambdas[support_vectors_id]
    targets = train_labels[support_vectors_id]

    b = np.sum(targets)
    for n in range(vector_number):
        b -= np.sum(lambdas * targets * np.reshape(kernel[support_vectors_id[n], support_vectors_id], (vector_number, 1)))
    b /= len(lambdas)

    return lambdas, support_vectors, support_vectors_id, b, targets, vector_number

def build_kernel(data_set, kernel_type='linear'):
    kernel = np.dot(data_set, data_set.T)
    if kernel_type == 'rbf':
        sigma = 1.0
        objects_count = len(data_set)
        b = np.ones((len(data_set), 1))
        kernel -= 0.5 * (np.dot((np.diag(kernel)*np.ones((1, objects_count))).T, b.T)
                         + np.dot(b, (np.diag(kernel) * np.ones((1, objects_count))).T.T))
        kernel = np.exp(kernel / (2. * sigma ** 2))
    return kernel

def classify_rbf(test_data_set, train_data_set, lambdas, targets, b, vector_number, support_vectors, support_vectors_id):
    kernel = np.dot(test_data_set, support_vectors.T)
    sigma = 1.0
    #K = np.dot(test_data_set, support_vectors.T)
    #kernel = build_kernel(train_data_set, kernel_type='rbf')
    c = (1. / sigma * np.sum(test_data_set ** 2, axis=1) * np.ones((1, np.shape(test_data_set)[0]))).T
    c = np.dot(c, np.ones((1, np.shape(kernel)[1])))
    #aa = np.dot((np.diag(K)*np.ones((1,len(test_data_set)))).T[support_vectors_id], np.ones((1, np.shape(K)[0]))).T
    sv = (np.diag(np.dot(train_data_set, train_data_set.T))*np.ones((1,len(train_data_set)))).T[support_vectors_id]
    aa = np.dot(sv,np.ones((1,np.shape(kernel)[0]))).T
    kernel = kernel - 0.5 * c - 0.5 * aa
    kernel = np.exp(kernel / (2. * sigma ** 2))

    y = np.zeros((np.shape(test_data_set)[0], 1))
    for j in range(np.shape(test_data_set)[0]):
        for i in range(vector_number):
            y[j] += lambdas[i] * targets[i] * kernel[j, i]
        y[j] += b
    return np.sign(y)

In [16]:
# modify this part 
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_iris

cvxopt.solvers.options['show_progress'] = True

iris = load_iris()
data_set = np.concatenate((iris.data, iris.target.reshape(-1, 1)), axis=1)
labels_count = get_labels_count(data_set)

_, test_data_set, _, true_test_labels = train_test_split(data_set[:, :-1], data_set[:, -1], test_size=0.2, random_state=42)

votes = np.zeros((len(test_data_set), labels_count))

for label in range(labels_count):
    train_data_set, _, train_labels, _ = choose_set_for_label(data_set, label)
    
    objects_count = len(train_data_set)

    lambdas, SVs, SV_ids, b, targets, vec_num = train(train_data_set, train_labels, kernel_type='rbf')
    
    predictions = classify_rbf(test_data_set, train_data_set, lambdas, targets, b, vec_num, SVs, SV_ids)
    
    votes[:, label] = predictions.flatten()

predicted = np.argmax(votes, axis=1)

print(f"Accuracy: {accuracy_score(predicted, true_test_labels)}")

     pcost       dcost       gap    pres   dres
 0:  1.3543e+02 -1.9835e+03  4e+03  2e-01  2e-15
 1:  8.6439e+01 -2.2486e+02  4e+02  9e-03  2e-15
 2:  1.2484e+01 -2.8119e+01  4e+01  2e-05  2e-15
 3: -7.5365e-01 -6.2783e+00  6e+00  2e-16  1e-15
 4: -2.0516e+00 -3.8076e+00  2e+00  2e-16  5e-16
 5: -2.5242e+00 -3.0976e+00  6e-01  2e-16  4e-16
 6: -2.7057e+00 -2.9450e+00  2e-01  1e-16  4e-16
 7: -2.7722e+00 -2.8528e+00  8e-02  2e-16  5e-16
 8: -2.7947e+00 -2.8064e+00  1e-02  2e-16  6e-16
 9: -2.7988e+00 -2.7992e+00  4e-04  3e-16  6e-16
10: -2.7989e+00 -2.7989e+00  4e-06  4e-16  5e-16
11: -2.7989e+00 -2.7989e+00  4e-08  6e-16  5e-16
Optimal solution found.
     pcost       dcost       gap    pres   dres
 0:  1.8559e+02 -4.6699e+03  9e+03  3e-01  4e-15
 1:  1.2358e+02 -5.4904e+02  7e+02  5e-03  4e-15
 2: -3.7544e+01 -2.0766e+02  2e+02  8e-04  4e-15
 3: -7.7104e+01 -1.2468e+02  5e+01  2e-04  5e-15
 4: -8.6869e+01 -1.1035e+02  2e+01  5e-05  4e-15
 5: -9.1449e+01 -1.0378e+02  1e+01  2e-05  5e-1